In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

def linear(Z):
    return Z

def compute_loss(Y, Y_hat):
    return np.mean((Y_hat - Y) ** 2)

df = pd.read_csv("/content/drive/MyDrive/abalone.csv")

df = pd.get_dummies(df, drop_first=True)

X = df.drop("Rings", axis=1).values.astype(float)
y = df["Rings"].values.reshape(-1, 1).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def forward_propagation(X, params):
    W1, b1 = params["W1"], params["b1"]
    W2, b2 = params["W2"], params["b2"]

    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = linear(Z2)

    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

def backward_propagation(Y, cache, params):
    m = Y.shape[0]
    X, Z1, A1, Z2, A2 = cache["X"], cache["Z1"], cache["A1"], cache["Z2"], cache["A2"]
    W2 = params["W2"]

    dZ2 = (A2 - Y) / m
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

np.random.seed(42)
input_dim = X_train.shape[1]
hidden_dim = 32
output_dim = 1

params = {
    "W1": np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim),
    "b1": np.zeros((1, hidden_dim)),
    "W2": np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim),
    "b2": np.zeros((1, output_dim))
}

learning_rate = 0.01

y_pred, cache = forward_propagation(X_train, params)
grads = backward_propagation(y_train, cache, params)

params["W1"] -= learning_rate * grads["dW1"]
params["b1"] -= learning_rate * grads["db1"]
params["W2"] -= learning_rate * grads["dW2"]
params["b2"] -= learning_rate * grads["db2"]

test_preds, _ = forward_propagation(X_test, params)

print("\nMSE =", mean_squared_error(y_test, test_preds))
print("R2 Score =", r2_score(y_test, test_preds))


MSE = 89.79902281042018
R2 Score = -7.295360776604218
